<a href="https://colab.research.google.com/github/NicolaDiCrescenzo04/Analisi_Statistica_Tesi_Triennale/blob/main/01_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **IMPORTO LIBRERIE NECESSARIE**

In [40]:
import pandas as pd
import numpy as np

### **IMPORTO I DATI**

In [41]:
#Importo dati da Google Drive (presenti nella cartella condivisa).
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
#Percorso cartella condivisa nel Drive (adattare al proprio Google Drive se necessario)
percorso_cartella = "/content/drive/MyDrive/Analisi_Statistica_DiCrescenzoNicola"

In [43]:
#Importo i fogli excel come dataframe per l'analisi
df_software = pd.read_excel(f"{percorso_cartella}/Datas/Società_Software.xlsm", sheet_name="Risultati")
df_hardware = pd.read_excel(f"{percorso_cartella}/Datas/Società_Hardware.xlsm", sheet_name="Risultati")

#Stampo dimensione campione iniziale
print("Dimensione campione iniziale: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")

Dimensione campione iniziale: 836 osservazioni.


### **PULISCO I DATI**

Rimuovo le righe che hanno valori mancanti

In [44]:
def pulizia_valori_mancanti(df):
  df = df.copy()

  """So che nel dataframe, almeno nella parte numerica, non ci sono celle bianche. Considero comunque in futuro di inserire
  #una funzione per rimuovere le celle vuote. """


  #Scelgo valori da rimuovere, in questo caso le celle non definite o non significative
  valori_da_rimuovere = ["n.d.", "n.s."]

  #Creo una maschera che restituisce True quando il valore della colonna è uguale ad uno dei valori da rimuovere
  maschera_valori_invalidi = (
      df.astype(str)
      .apply(lambda x:
             x.str.strip().str.lower())
      .isin(valori_da_rimuovere)
      .any(axis=1)
  )

  #Escludiamo le righe True dal Dataframe
  df = df[~maschera_valori_invalidi]
  return df


df_software = pulizia_valori_mancanti(df_software)
df_hardware = pulizia_valori_mancanti(df_hardware)

#Stampo la dimensione del campione statistico a disposizione dopo la pulizia
print("Dimensione campione software pulito:" + str(len(df_software)) + "osservazioni")
print("Dimensione campione hardware pulito:" + str(len(df_hardware)) + "osservazioni")
print("Dimensione campione totale pulito: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")



Dimensione campione software pulito:276osservazioni
Dimensione campione hardware pulito:86osservazioni
Dimensione campione totale pulito: 362 osservazioni.


Converto correttamente le colonne numeriche

In [45]:
colonne_numeriche = [
    "Numero dipendenti\nUltimo anno disp.",

    "Redditività del capitale proprio (ROE) - Netto\n2024",
    "Redditività del capitale proprio (ROE) - Netto\n2023",
    "Redditività del capitale proprio (ROE) - Netto\n2022",
    "Redditività del capitale proprio (ROE) - Netto\n2021",
    "Redditività del capitale proprio (ROE) - Netto\n2020",

    "Indice di liquidità\n2024",
    "Indice di liquidità\n2023",
    "Indice di liquidità\n2022",
    "Indice di liquidità\n2021",
    "Indice di liquidità\n2020",

    "Debt/Equity\n2024",
    "Debt/Equity\n2023",
    "Debt/Equity\n2022",
    "Debt/Equity\n2021",
    "Debt/Equity\n2020",

    "Totale Attivo\nmigl USD 2024",
    "Totale Attivo\nmigl USD 2023",
    "Totale Attivo\nmigl USD 2022",
    "Totale Attivo\nmigl USD 2021",
    "Totale Attivo\nmigl USD 2020",
]

#Creo una funzione per convertire le colonne numeriche correttamente.
def converti_colonne_numeriche(df, colonne):
  df = df.copy()

#Sostituisco i punti (separatori migliaia) e le virgole (separatori decimali) per uniformarli.
  for colonna in colonne:
    df[colonna] = (
        df[colonna]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
        .str.replace(".", "", regex=False)
        .astype(float)
    )
    df[colonna] = pd.to_numeric(df[colonna], errors="coerce")
  return df

df_hardware = converti_colonne_numeriche(df_hardware, colonne_numeriche)
df_software = converti_colonne_numeriche(df_software, colonne_numeriche)

Converto correttamente le colonne con date

In [46]:
colonne_date = [
    "Data chiusura\n2024",
    "Data chiusura\n2023",
    "Data chiusura\n2022",
    "Data chiusura\n2021",
    "Data chiusura\n2020",
    "Data di costituzione",
]

def converti_colonne_date(df, colonne):
    df = df.copy()
    for colonna in colonne:
        serie = (
            df[colonna]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        df[colonna] = pd.to_datetime(
            serie,
            errors="coerce",
            dayfirst=False
        )

    return df

df_hardware = converti_colonne_date(df_hardware, colonne_date)
df_software = converti_colonne_date(df_hardware, colonne_date)

Trasformo i "Total assets" in valori logaritmici per normalizzare le distribuzione

In [47]:
colonne_attivo = [
    "Totale Attivo\nmigl USD 2024",
    "Totale Attivo\nmigl USD 2023",
    "Totale Attivo\nmigl USD 2022",
    "Totale Attivo\nmigl USD 2021",
    "Totale Attivo\nmigl USD 2020",
]


def log_attivo(df, colonne_attivo):
    df = df.copy()

    for colonna in colonne_attivo:
        nuova_colonna = "log_" + colonna

        # Il log naturale è definito solo per valori > 0
        df[nuova_colonna] = np.where(
            df[colonna] > 0,
            np.log(df[colonna]),
            np.nan
        )

    return df

df_hardware = log_attivo(df_hardware, colonne_attivo)
df_software = log_attivo(df_software, colonne_attivo)